In [ ]:
from functools import partial
from pathlib import Path
import os
import shutil

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ─────────────────────────────────────────────────────────────────────────────

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-2)
num_usable_cores = max(1, (os.cpu_count() or 2) - 2)
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

# new_weather_epw = "Lecco_IT_TMY"
# new_weather_climate_zone = "4A"

### Activity 3: REopt

In [ ]:
# This is the same as above, but in a new directory
activity_3_analysis_dir = analysis_dir / "activity_3"
if not activity_3_analysis_dir.exists():
    activity_3_analysis_dir.mkdir()

uo_coincident = UO(
    activity_3_analysis_dir, "coincident_pre", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

uo_coincident.create_scenarios("class_project_coincident.json")

# Copy over to the coincident directory for activity 3
uo_coincident = uo_coincident.update_project_files("coincident")

# Run sims faster
uo_coincident.set_number_parallel(num_usable_cores)

# Copy over the weather files
uo_coincident.copy_over_weather()

# Change weather file in feature + mapper files when enabled
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

# Fix issues -- copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
# Copy over the base workflow from the template. This one includes all the EEMs that are needed in the class project since
# the class_project_workflow.json inherits from base_workflow.osw. This is a workaround to avoid having to patch the workflow file to add in the missing EEM steps.
shutil.copy2(
    template_data_dir / "mappers" / "base_workflow.osw",
    uo_coincident.project_path / "mappers" / "base_workflow.osw",
)

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# Post-process and visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
# Create the scenario mapper file that is enabled with the REopt assumptions
uo_coincident.create_reopt_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)

# Confirm in the REopt_baseline_scenario file that the REopt assumptions are now enabled (look for the REopt Assumptions section) in the file

In [ ]:
# Run the REopt baseline scenario
uo_coincident.run("class_project_coincident.json", "REopt_baseline_scenario.csv")

In [ ]:
import copy
import json
import os
import shutil

reopt_dir = activity_3_analysis_dir / "coincident" / "reopt"
base_assumptions_path = reopt_dir / "multiPV_assumptions.json"

# -----------------------------------------------------------------------
# 1.  Load base assumptions and apply the changes
# -----------------------------------------------------------------------
with open(base_assumptions_path) as f:
    base = json.load(f)

optimized = copy.deepcopy(base)

# --- Financial -----------------------------------------------------------
optimized["Financial"]["elec_cost_escalation_rate_fraction"] = 0.05  # 0.026 → 0.05
optimized["Financial"]["offtaker_discount_rate_fraction"] = 0.04  # 0.081 → 0.04
optimized["Financial"]["emissions_cost_per_mt_co2"] = 100.0  # add carbon tax

# --- PV ------------------------------------------------------------------
for pv_array in optimized["PV"]:
    pv_array["installed_cost_per_kw"] = 1000.0  # $2,000 → $1,000 per kW

# --- ElectricTariff: $25/kW monthly demand charge ------------------------
optimized["ElectricTariff"]["add_monthly_rates_to_urdb_rate"] = True
optimized["ElectricTariff"]["monthly_demand_rates"] = [25.0] * 12

# --- ElectricUtility: allow export so PV surplus can charge battery ------
# Without this, REopt curtails midday PV instead of storing it.
optimized["ElectricUtility"]["net_metering_limit_kw"] = 1_000_000.0
optimized["ElectricUtility"]["interconnection_limit_kw"] = 1_000_000.0

# --- ElectricStorage: enable ITC + refresh costs to ATB 2024 defaults ----
optimized["ElectricStorage"].update(
    {
        "min_kw": 0.0,
        "max_kw": 1_000_000.0,
        "min_kwh": 0.0,
        "max_kwh": 1_000_000.0,
        "installed_cost_per_kw": 775.0,  # power-component cost
        "installed_cost_per_kwh": 388.0,  # energy-component cost (4-hr Li-ion)
        "replace_cost_per_kw": 410.0,
        "replace_cost_per_kwh": 220.0,
        "battery_replacement_year": 10,
        "inverter_replacement_year": 10,
        "total_itc_fraction": 0.3,  # 0.0 → 0.3
        "macrs_option_years": 7,
        "macrs_bonus_fraction": 0.6,
        "macrs_itc_reduction": 0.5,
        "soc_min_fraction": 0.2,
        "soc_init_fraction": 0.5,
        "internal_efficiency_fraction": 0.975,
        "inverter_efficiency_fraction": 0.96,
        "rectifier_efficiency_fraction": 0.96,
        "can_grid_charge": True,
    }
)

# --- Wind: replace the all-zero block with a real distributed-wind setup -
# "size_class" controls turbine archetype: residential | commercial | medium | large
optimized["Wind"] = {
    "min_kw": 0.0,
    "max_kw": 1_000_000.0,
    "size_class": "commercial",
    "installed_cost_per_kw": 3013.0,  # ATB 2024 distributed wind, commercial
    "om_cost_per_kw": 40.0,
    "macrs_option_years": 5,
    "macrs_bonus_fraction": 0.6,
    "federal_itc_fraction": 0.3,
    "can_net_meter": True,
    "can_wholesale": False,
    "can_export_beyond_nem_limit": False,
}

# -----------------------------------------------------------------------
# 2.  Write the new assumptions file
# -----------------------------------------------------------------------
new_assumptions_name = "multiPV_optimized_assumptions.json"
new_assumptions_path = reopt_dir / new_assumptions_name
with open(new_assumptions_path, "w") as f:
    json.dump(optimized, f, indent=2)
print(f"Wrote: {new_assumptions_path}")

# -----------------------------------------------------------------------
# 3.  Build new scenario CSV pointing to the new assumptions file
# -----------------------------------------------------------------------
base_mapper = activity_3_analysis_dir / "coincident" / "REopt_baseline_scenario.csv"
new_mapper = activity_3_analysis_dir / "coincident" / "REopt_optimized_scenario.csv"

mapper_text = base_mapper.read_text()
new_csv_text = mapper_text.replace("multiPV_assumptions.json", new_assumptions_name)
new_mapper.write_text(new_csv_text)
print(f"Wrote: {new_mapper}")

# -----------------------------------------------------------------------
# 4.  Summary of parameter changes
# -----------------------------------------------------------------------
print("\nParameter changes applied:")
print(
    f"  Financial.elec_cost_escalation_rate   : {base['Financial']['elec_cost_escalation_rate_fraction']} → {optimized['Financial']['elec_cost_escalation_rate_fraction']}"
)
print(
    f"  Financial.offtaker_discount_rate      : {base['Financial']['offtaker_discount_rate_fraction']} → {optimized['Financial']['offtaker_discount_rate_fraction']}"
)
print(
    f"  Financial.emissions_cost_per_mt_co2   : {optimized['Financial']['emissions_cost_per_mt_co2']}"
)
print(
    f"  PV[*].installed_cost_per_kw           : {base['PV'][0]['installed_cost_per_kw']} → {optimized['PV'][0]['installed_cost_per_kw']}"
)
print(
    f"  ElectricTariff.monthly_demand_rates   : ${optimized['ElectricTariff']['monthly_demand_rates'][0]}/kW × 12"
)
print(
    f"  ElectricUtility.net_metering_limit_kw : {base['ElectricUtility']['net_metering_limit_kw']} → {optimized['ElectricUtility']['net_metering_limit_kw']:,.0f}"
)
print(
    f"  ElectricStorage.total_itc_fraction    : {base['ElectricStorage']['total_itc_fraction']} → {optimized['ElectricStorage']['total_itc_fraction']}"
)
print(
    f"  ElectricStorage.installed_cost_per_kw : {base['ElectricStorage']['installed_cost_per_kw']} → {optimized['ElectricStorage']['installed_cost_per_kw']}"
)
print(
    f"  ElectricStorage.installed_cost_per_kwh: {base['ElectricStorage']['installed_cost_per_kwh']} → {optimized['ElectricStorage']['installed_cost_per_kwh']}"
)
print(
    f"  Wind.max_kw                           : {base['Wind']['max_kw']} → {optimized['Wind']['max_kw']:,.0f}"
)
print(
    f"  Wind.installed_cost_per_kw            : {base['Wind']['installed_cost_per_kw']} → {optimized['Wind']['installed_cost_per_kw']}"
)
print(f"  Wind.size_class                       : {optimized['Wind']['size_class']}")

In [ ]:
# copy already run scenario to save simulation time

shutil.copytree(
    activity_3_analysis_dir / "coincident" / "run" / "reopt_baseline_scenario",
    activity_3_analysis_dir / "coincident" / "run" / "reopt_optimized_scenario",
    dirs_exist_ok=True,
)

In [ ]:
# Post-process the optimized REopt scenario (scenario-level summary)
os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

# baseline
uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=False,
)

In [ ]:
# Post-process the optimized REopt scenario (scenario-level summary)
os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

# optimized
uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_optimized_scenario.csv",
    individual_features=False,
)

In [ ]:
# Force embedded Ruby/URBANopt to use a valid CA bundle on an NLR macOS.
os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=False,
)

In [ ]:
# THIS TAKES A LONG TIME TO RUN (+/- 3min*23buildings =~70 minutes)
# Force embedded Ruby/URBANopt to use a valid CA bundle on an NLR macOS.
os.environ.setdefault("SSL_CERT_FILE", "/etc/ssl/cert.pem")
print(f"Using SSL_CERT_FILE={os.environ['SSL_CERT_FILE']}")

# run individual features to get the breakdown of the different features in the REopt scenario
uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=True,
)